# Chapter 6 — Indexing and Offset Calculations

Part 2: Pointer arithmetic and memory operations.

**Thread.**

- **Thread.** Every address in the chapter is one dot product: coordinate times element stride, plus the base.
- **Thread.** 1-D adds a program id and a lane index; 2-D adds a row stride; 3-D adds a plane stride.
- **Thread.** Flattening collapses a coordinate to one integer; recovering the coordinate inverts that map.
- **Thread.** Strides replace hard-coded widths, so one kernel serves contiguous, transposed, and sliced inputs.
- **Thread.** Tiling and masks only change how the index vectors are built, never the address formula itself.

**Ladder:** host-side index arithmetic and Python loops (Method 01) → Triton offset tiles, 2-D tile copy, and row/column gather (Method 02) → 3-D address recovery, stride arguments, row-wise scale, and `tl.static_range` (Method 03).

**Study tip.**

- Fix a $3\times4$ matrix whose values are $0,\ldots,11$.
- Write $f(i,j)=iN+j=4i+j$ and its inverse $i=\lfloor f/4\rfloor$, $j=f\bmod 4$.
- Check $f(1,2)=6$ and $f=6\mapsto(1,2)$ by hand.
- Fill in the flat-index table in the worked example before reading any kernel.
- If those entries match, the rest of the chapter is implementation detail.

**Learning objectives**

- Build 1-D, 2-D, and 3-D offset maps, and state the shape of each offset tile.
- Convert between a multidimensional index and a flat index in both directions.
- Distinguish an element offset from a byte offset, and drive an address with `tensor.stride()`.
- Build tiled offsets with `rows[:, None]` and `cols[None, :]`, and state the `(BM, BN)` mask shape.
- Write a kernel whose address is recovered from a flat program id (row, column, and 3-D cases).
- Recognize the failure modes: swapped axes, off-by-one tails, wrong mask rank, int32 overflow, and non-contiguous strides.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch05_load_store/05_complete.ipynb) · [Next →](../../part3_math/ch07_arithmetic/07_complete.ipynb)

**Estimated time:** 60–75 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 4 (pointers, element versus byte offsets, row-major layout) and Chapter 5 (`tl.load`, `tl.store`, masks, `other=`); modular arithmetic.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter map

- Each row is one book_spec section.
- The fourth column is the shape or stride rule the kernel must satisfy.
- The fifth column names the runnable demo later in the chapter.

| § | Topic | Key formula | Shape/stride rule | Kernel/demo |
|---|---|---|---|---|
| **6.1** | 1-D offsets | $o_i=\operatorname{pid}\cdot B+i$ | one program owns $B$ consecutive lanes; mask $o_i\lt N$ | 1-D offset map; 2-D tile copy |
| **6.2** | 2-D offsets | $o(i,j)=i\,s_0+j\,s_1=iN+j$ | row-major $(M,N)$ has strides $(N,1)$ | 2-D tile copy |
| **6.3** | 3-D offsets | $o(i,j,k)=(iD_1+j)D_2+k$ | strides $(D_1D_2,D_2,1)$ | 3-D copy kernel |
| **6.4** | Row indexing | $o_{\text{row}}(i,j)=iN+j$ | fix $i$; $N$ consecutive lanes, stride 1 | row gather; row-wise scale |
| **6.5** | Column indexing | $o_{\text{col}}(i,j)=iN+j$ | fix $j$; $M$ lanes, stride $N$ | column gather |
| **6.6** | Flattening a matrix | $f(i,j)=iN+j$ | C-order; contiguous `reshape(-1)` | host flatten |
| **6.7** | Recovering row and column | $i=\lfloor f/N\rfloor,\;j=f\bmod N$ | integer div/mod by the row width | host unflatten; 3-D copy |
| **6.8** | Stride-based indexing | $o=i\,s_0+j\,s_1$ | $s_k$ in elements, not bytes | stride-argument copy |
| **6.9** | Broadcasting offsets | $O=\text{rows}\,s_0+\text{cols}\,s_1$ | $(B_M,1)+(1,B_N)\to(B_M,B_N)$ | 2-D tile copy |
| **6.10** | Tiled offsets | $(p_mB_M+m)N+(p_nB_N+n)$ | grid $(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$ | 2-D tile copy |
| **6.11** | Boundary masks | $[\,m\lt M\,]\land[\,n\lt N\,]$ | mask shape equals offset shape $(B_M,B_N)$ | 2-D tile copy |
| **6.12** | Arbitrary dimensions | $o=\sum_k i_k s_k$ | one `arange` per axis, expanded to rank | `tl.static_range` offsets |
| **★** | Capstone view | $o=f^{-1}(f)$ round trip | same shape, same strides | all demos together |

**One-line thesis.**

- An index is not a coordinate and a coordinate is not an address.
- The address is the dot product between the coordinate and the strides.
- Every kernel in this chapter is one way to write that dot product.

## Concept diagrams

### 1-D, 2-D, and 3-D offset maps

- `pid * B + lane` partitions a flat array into consecutive blocks.
- The 2-D map `i * N + j` walks a row-major matrix row by row.
- The 3-D map `(i * D1 + j) * D2 + k` walks one plane at a time.

```text
1-D:  offset = pid * B + lane
      pid 0            pid 1
      [0 1 2 3]        [4 5 6 7]        B = 4

2-D:  offset(i, j) = i * N + j                       N = 4
             j=0  j=1  j=2  j=3
        i=0   0    1    2    3
        i=1   4    5    6    7
        i=2   8    9   10   11

3-D:  offset(i, j, k) = (i * D1 + j) * D2 + k        D0=2, D1=3, D2=4
        plane i=0:  0 .. 11        plane i=1: 12 .. 23
        within a plane: row j moves by D2 = 4, column k by 1
```

### Flatten and unflatten

- Flatten sends $(i,j)$ to $f=iN+j$.
- Unflatten sends $f$ back with $i=f//N$ and $j=f\bmod N$.
- For the $3\times4$ case, $(1,2)\mapsto6\mapsto(1,2)$ and $(2,0)\mapsto8\mapsto(2,0)$.

```text
     (i, j)  --f = i*N + j-->  f  --i = f//N, j = f%N-->  (i, j)
      3x4 matrix                  flat buffer                 coordinate

        (1, 2)  ---------------->  1*4 + 2 = 6  ---------------->  (1, 2)
        (2, 0)  ---------------->  2*4 + 0 = 8  ---------------->  (2, 0)
```

### Tiled offsets

- A tile of shape $(B_M,B_N)$ is indexed by two program ids.
- The grid is $(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$.
- Edge tiles own lanes outside the matrix; those lanes are masked.

```text
matrix M=5, N=7, tile BM=2, BN=4        grid = (ceil(5/2), ceil(7/4)) = (3, 2)

      col 0..3      col 4..7
row 0 [ tile (0,0) ][ tile (0,1) ]
row 1 [            ][            ]
row 2 [ tile (1,0) ][ tile (1,1) ]
row 3 [            ][            ]
row 4 [ tile (2,0) ][ tile (2,1) ]
row 5 [  masked    ][  masked    ]
                        ^ col 7 is masked in every tile of the right column
```

### Boundary mask

- The mask is a rank-2 tile with the same shape as the offset tile.
- It guards both the load and the store.
- Address arithmetic still runs on dead lanes; only memory access is suppressed.

```text
pid=(2,1), rows=4,5, cols=4,5,6,7, M=5, N=7

 rows[:,None] < 5        cols[None,:] < 7        mask = both
 [1]                     [1 1 1 0]              [1 1 1 0]
 [0]                                             [0 0 0 0]
```

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

# Part I — Mathematics and concepts

- **Thread.** Every rank uses the same dot product: $o=\text{base}+\sum_k i_k s_k$.
- **Rule.** $s_k$ is the stride of axis $k$ in elements; a 1-D kernel keeps only $i_0$, a 2-D kernel keeps $(i,j)$, and a 3-D kernel keeps $(i,j,k)$.
- **Where.** The formula does not change with rank, only the number of terms does.
- **Bridge.** The twelve sections below follow book_spec order, each with its formula, use, and failure mode.

### 6.1 1-D offsets

- **Connection.** Builds on Chapter 5's pointer-plus-offset form and the flat program grid from Chapter 2.
- **Rule.** A 1-D program owns a contiguous block of $B$ lanes; the grid has $P=\lceil N/B\rceil$ programs.
- **Formula.** $o_i=\operatorname{pid}\cdot B+i$ for $i\in\{0,\ldots,B-1\}$, with $\text{mask}_i=[o_i\lt N]$.
- **Example.** With $B=4$, pid 0 owns $\{0,1,2,3\}$, pid 1 owns $\{4,5,6,7\}$, and pid 2 owns $\{8,9,10,11\}$.
- **Where.** This is the default layout for a flat array and the base case for every later rank.
- **Triton.** One `pid`, one `tl.arange(0, B)`, one add.
- **Failure.** Dropping `pid * B` repeats the first tile; `pid + i * B` interleaves tiles; `tl.arange` also requires a power-of-two length.
- **Bridge.** 6.2 adds a second index and turns the lane vector into a row of a matrix.

### 6.2 2-D offsets

- **Connection.** Builds on 6.1 by adding a second coordinate.
- **Rule.** A tensor of shape $(M,N)$ stored row-major has element strides $(N,1)$.
- **Formula.** $o(i,j)=i\,s_0+j\,s_1=iN+j$.
- **Example.** For the $3\times4$ matrix, $o(1,2)=6$ and the offset tile is `[[0,1,2,3],[4,5,6,7],[8,9,10,11]]`.
- **Where.** Collecting every pair gives an $(M,N)$ offset tile; the offset is an element index, not a byte index.
- **Triton.** A 2-D grid maps `tl.program_id(0)` to rows and `tl.program_id(1)` to columns; `x_ptr + o` already scales by the element size.
- **Failure.** Swapping $i$ and $j$ transposes the result silently; hard-coding $N$ breaks transposed or sliced inputs, so use the actual strides (6.8).
- **Bridge.** 6.3 adds a third axis for batched matrices and images.

### 6.3 3-D offsets

- **Connection.** Builds on 6.2 by adding an outer axis.
- **Rule.** For shape $(D_0,D_1,D_2)$ with contiguous row-major strides $(D_1D_2,D_2,1)$, the outer index moves by a whole plane, the middle by a row, and the inner by one element.
- **Formula.** $o(i,j,k)=(iD_1+j)D_2+k=iD_1D_2+jD_2+k$.
- **Example.** For $(2,3,4)$, $o(1,2,3)=23$ and $o(0,1,2)=6$; plane $i=0$ covers offsets $0,\ldots,11$ and plane $i=1$ covers $12,\ldots,23$.
- **Where.** Batched matrices, image tensors, and attention heads are rank-3; recovering $(i,j,k)$ from a flat lane also gives a generic copy kernel.
- **Triton.** A 3-D grid uses the same idea with `tl.program_id(0..2)`.
- **Failure.** Wrong nesting, such as $iD_0D_1+\cdots$, reads the wrong plane; in a contiguous tensor the inner dimension must have stride 1.
- **Bridge.** 6.4 fixes one index to read a row.

### 6.4 Row indexing

- **Connection.** Fixes the row index $i$ in 6.2.
- **Rule.** A row is contiguous, so its lanes have stride 1.
- **Formula.** $o_{\text{row}}(i,j)=iN+j$ for $j=0,\ldots,N-1$, that is, $iN+\{0,\ldots,N-1\}$.
- **Example.** Row 2 of the $3\times4$ matrix has offsets $\{8,9,10,11\}$ and values $\{8,9,10,11\}$.
- **Where.** Row normalisation, row gather, and row softmax all need one program per row.
- **Triton.** `base + i * N + tl.arange(0, N)` gives one scalar base and one contiguous lane vector.
- **Failure.** `base + i + tl.arange(0, N)` indexes a diagonal-like neighbourhood; a column-major row must use stride $s_1$, not 1.
- **Bridge.** 6.5 fixes the other index and reads a column instead.

### 6.5 Column indexing

- **Connection.** Fixes the column index $j$ in 6.2 and is the dual of 6.4.
- **Rule.** A column has stride $N$, so its lanes are $N$ elements apart.
- **Formula.** $o_{\text{col}}(i,j)=iN+j$ for $i=0,\ldots,M-1$, that is, $j+\{0,N,2N,\ldots\}$.
- **Example.** Column 3 of the $3\times4$ matrix has offsets $\{3,7,11\}$ and values $\{3,7,11\}$.
- **Where.** Column gather, feature-wise scaling, and transposed access patterns use this formula; it is correct but touches one element per cache line.
- **Triton.** `base + j + tl.arange(0, M) * N` is the whole kernel address.
- **Failure.** `base + j + tl.arange(0, M)` addresses adjacent elements of row 0, not column $j$; the stride $N$ is the point of the formula.
- **Bridge.** 6.6 drops both coordinates and keeps only a flat index.

### 6.6 Flattening a matrix

- **Connection.** Combines the two indices of 6.2 into one integer.
- **Rule.** Row-major flattening follows C-order; `A.reshape(-1)` returns that order for a contiguous tensor, while `view(-1)` raises on a non-contiguous one.
- **Formula.** $f(i,j)=iN+j$ for $f\in\{0,\ldots,MN-1\}$.
- **Example.** The $3\times4$ matrix flattens to $(0,1,\ldots,11)$ and $f(1,2)=6$.
- **Where.** A flat index is the program id: a 1-D grid cannot see $(i,j)$, so it either recovers them (6.7) or the host passes a precomputed index table.
- **Triton.** `tl.load(ptr + offs)` treats a matrix as a flat buffer with the same mapping.
- **Failure.** Flattening a non-contiguous tensor silently copies; a column-major flatten uses $f=jM+i$, not $iN+j$.
- **Bridge.** 6.7 inverts this flat map.

### 6.7 Recovering row and column

- **Connection.** Inverts the flattening of 6.6.
- **Rule.** Use quotient and remainder by the row width $N$; no floating point is involved.
- **Formula.** $i=\lfloor f/N\rfloor$ and $j=f\bmod N$, written `f // N` and `f % N`.
- **Example.** $f=6\mapsto(1,2)$ and $f=8\mapsto(2,0)$.
- **Where.** A 1-D copy or gather receives only a flat lane; dividing by the row width recovers the coordinate, which then drives a possibly non-contiguous address.
- **Triton.** In 3-D the same trick divides twice: $i=f//(D_1D_2)$, $j=(f\bmod D_1D_2)//D_2$, $k=f\bmod D_2$.
- **Failure.** The wrong width maps every index to the wrong row; `/` produces a float; swapping quotient and remainder transposes the recovery.
- **Bridge.** 6.8 replaces the hard-coded width with explicit strides.

### 6.8 Stride-based indexing

- **Connection.** Generalises 6.2 and 6.6 from contiguous storage to arbitrary layout.
- **Rule.** Strides are measured in elements, not bytes; a zero stride is a broadcast.
- **Formula.** $o(i,j)=i\,s_0+j\,s_1$.
- **Example.** The $3\times4$ matrix has $(s_0,s_1)=(4,1)$; its transpose has $(1,4)$ and offset $(2,1)=2\cdot1+1\cdot4=6$; byte strides are $(16,4)$.
- **Where.** Passing `x.stride(0)` and `x.stride(1)` lets one kernel serve contiguous, transposed, and sliced inputs.
- **Triton.** The kernel receives the two strides as scalar arguments and forms the address explicitly.
- **Failure.** Confusing a byte stride with an element stride multiplies every address by 4 for `float32`; a zero stride is legal but not a copy.
- **Bridge.** 6.9 combines two strided index vectors into a full tile.

### 6.9 Broadcasting offsets

- **Connection.** Combines the lane vector of 6.1 with the stride formula of 6.8.
- **Rule.** `rows[:, None]` has shape $(B_M,1)$ and `cols[None, :]` has shape $(1,B_N)$; `[None, :]` adds a dimension and does not copy data.
- **Formula.** $O=\text{rows}\,s_0+\text{cols}\,s_1\in\mathbb Z^{B_M\times B_N}$.
- **Example.** $(3,1)+(1,4)\to(3,4)$; `rows + cols` gives `[[0,1,2,3],[1,2,3,4],[2,3,4,5]]`, and `rows*4 + cols` gives the $3\times4$ offset tile.
- **Where.** This is the single most useful line in a 2-D kernel: two `tl.arange` vectors become a full tile of addresses.
- **Triton.** `tl.expand_dims(x, axis)` is the same rank change written as a function; the kernel still needs the strides passed explicitly.
- **Failure.** `rows + cols` with two rank-1 vectors is a vector add, not an outer sum; it is silent when the lengths match and raises when they differ.
- **Bridge.** 6.10 places this tile on a 2-D program grid.

### 6.10 Tiled offsets

- **Connection.** Combines 6.2 and 6.9 with the program ids of Chapter 2.
- **Rule.** Every program works on a tile of the same shape, so the kernel body does not depend on the tile's position.
- **Formula.** $O(p_m,p_n,m,n)=(p_mB_M+m)N+(p_nB_N+n)$, with grid $(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$.
- **Example.** For $M=5$, $N=7$, $B_M=2$, $B_N=4$ the grid is $(3,2)$; pid $(1,1)$ owns rows 2,3, cols 2,3 and offsets `[[10,11],[14,15]]`.
- **Where.** Tiling keeps a 2-D kernel within a fixed register budget, gives coalesced accesses, and is the layout of a tiled matrix multiply.
- **Triton.** `tl.program_id(0)` selects the row tile and `tl.program_id(1)` selects the column tile.
- **Failure.** One `pid` for both axes makes the tile diagonal; swapping the program ids transposes the grid; `M // BM` drops the partial last tile.
- **Bridge.** 6.11 handles the dead lanes of the edge tiles.

### 6.11 Boundary masks

- **Connection.** Completes 6.10 when $M$ or $N$ is not a multiple of the tile size.
- **Rule.** The mask has the same shape as the offset tile, $(B_M,B_N)$, and guards both the load and the store.
- **Formula.** $\text{mask}(m,n)=[\,p_mB_M+m\lt M\,]\land[\,p_nB_N+n\lt N\,]$.
- **Example.** For $M=5$, $N=7$, $B_M=2$, $B_N=4$, pid $(2,1)$ has mask `[[1,1,1,0],[0,0,0,0]]` and pid $(2,0)$ has mask `[[1,1,1,1],[0,0,0,0]]`.
- **Where.** The last tile has dead lanes; the mask keeps them from touching memory.
- **Triton.** `(rows[:,None] < M) & (cols[None,:] < N)` uses `&`, not `and`, and `<`, not `<=`.
- **Failure.** A rank-1 mask broadcasts along one axis and lets the other run out of bounds; `and` raises on a tensor; `<=` keeps one extra lane.
- **Bridge.** 6.12 repeats the same pattern for any rank.

### 6.12 Arbitrary dimensions

- **Connection.** Generalises 6.1–6.11 from ranks 1, 2, and 3 to rank $r$.
- **Rule.** Build one `tl.arange` per dimension, expand each to the output rank, multiply by its stride, and add.
- **Formula.** $o(i_0,\ldots,i_{r-1})=\text{base}+\sum_{k=0}^{r-1} i_k s_k$.
- **Example.** Rank 3 with $(2,3,4)$ uses strides $(12,4,1)$ and produces offsets $0,\ldots,23$; a `static_range` over $D_0$ writes one plane per iteration.
- **Where.** The pattern is the same at every rank; only the number of terms grows, so one copy or normalisation kernel can serve ranks 2, 3, and 4.
- **Triton.** `for k in tl.static_range(0, r)` unrolls the sum at compile time, so the bound must be a `tl.constexpr`.
- **Failure.** Mixing the axis order between shape and stride list reads the wrong element; a rank mismatch broadcasts silently or raises; a runtime bound in `static_range` raises.
- **Bridge.** Part II turns these formulas into host-side references before any Triton kernel appears.

In [ ]:
# Part I in one cell: every number stated in 6.1--6.12 is asserted here.
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
M, N = A.shape
rows = torch.arange(M)[:, None]      # (3, 1)
cols = torch.arange(N)[None, :]      # (1, 4)
offs2 = rows * N + cols              # (3, 4)

assert A.shape == (3, 4) and A.stride() == (4, 1)
assert tuple(s * A.element_size() for s in A.stride()) == (16, 4)
assert A.tolist() == [[0.0, 1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0, 7.0],
                      [8.0, 9.0, 10.0, 11.0]]

# 6.1, 6.2, 6.4, 6.5, 6.6, 6.7
assert (0 * 4 + torch.arange(4)).tolist() == [0, 1, 2, 3]
assert (1 * 4 + torch.arange(4)).tolist() == [4, 5, 6, 7]
assert (2 * 4 + torch.arange(4)).tolist() == [8, 9, 10, 11]
assert offs2.int().tolist() == A.int().tolist()
assert (2 * N + torch.arange(N)).tolist() == [8, 9, 10, 11]
assert (torch.arange(M) * N + 3).tolist() == [3, 7, 11]
assert A.reshape(-1).tolist() == [float(x) for x in range(12)]
assert (6 // N, 6 % N) == (1, 2)
assert (8 // N, 8 % N) == (2, 0)

# 6.8: element strides, a transpose, and a zero-stride broadcast view.
T = A.t()                            # shape (4, 3), strides (1, 4)
assert T.shape == (4, 3) and T.stride() == (1, 4)
assert 2 * T.stride(0) + 1 * T.stride(1) == 6 and T[2, 1] == 6
expanded = torch.arange(3, dtype=torch.float32)[:, None].expand(3, 4)
assert 0 in expanded.stride()
assert torch.equal(expanded[:, 0], expanded[:, 3])

# 6.9, 6.10, 6.11: shapes, tile offsets, and the rank-2 mask.
assert rows.shape == (3, 1) and cols.shape == (1, 4)
assert offs2.shape == (3, 4)
pid_m, pid_n, BM, BN = 1, 1, 2, 2
tile_rows = pid_m * BM + torch.arange(BM)
tile_cols = pid_n * BN + torch.arange(BN)
tile_offs = tile_rows[:, None] * N + tile_cols[None, :]
tile_mask = (tile_rows[:, None] < M) & (tile_cols[None, :] < N)
assert tile_rows.tolist() == [2, 3] and tile_cols.tolist() == [2, 3]
assert tile_offs.tolist() == [[10, 11], [14, 15]]
assert tile_mask.tolist() == [[True, True], [False, False]]

# 6.3 and 6.12: the rank-3 offset is the nested dot product.
D0, D1, D2 = 2, 3, 4
assert (1 * D1 + 2) * D2 + 3 == 23
assert [(i * D1 * D2 + j * D2 + k)
        for i in range(D0) for j in range(D1) for k in range(D2)] == list(range(24))
plane = lambda i: i * D1 * D2 + torch.arange(D1)[:, None] * D2 + torch.arange(D2)[None, :]
assert plane(0).tolist() == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
assert plane(1).tolist() == [[12, 13, 14, 15], [16, 17, 18, 19], [20, 21, 22, 23]]

## Worked numerical example

- **Connection.** Applies the maps from 6.1–6.7 to one concrete matrix.
- **Example.** Let $A$ be the $3\times4$ matrix below with values $0,\ldots,11$ and $N=4$.

$$
A=\begin{bmatrix}
0&1&2&3\\
4&5&6&7\\
8&9&10&11
\end{bmatrix}
$$

- **Flatten.** $f(i,j)=4i+j$, so $f(1,2)=4+2=6$ and $f(2,0)=8$; its inverse is $i=f//4$, $j=f\bmod 4$, hence $6\mapsto(1,2)$ and $8\mapsto(2,0)$.
- **Rows.** $o_{\text{row}}(2,j)=2\cdot4+j=8+j$, so row 2 has offsets $(8,9,10,11)$ and values $(8,9,10,11)$.
- **Columns.** $o_{\text{col}}(i,3)=4i+3$, so column 3 has offsets $(3,7,11)$ and values $(3,7,11)$.
- **Inverse table.** The flat buffer is $(0,1,\ldots,11)$ and every flat index maps back as follows.

| $f$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| row $f//4$ | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 | 2 | 2 | 2 | 2 |
| col $f\bmod4$ | 0 | 1 | 2 | 3 | 0 | 1 | 2 | 3 | 0 | 1 | 2 | 3 |

- **3-D case.** For $T$ of shape $(2,3,4)$ with values $0,\ldots,23$ and strides $(12,4,1)$, $o(1,2,3)=(1\cdot3+2)\cdot4+3=23$ and $o(0,1,2)=(0\cdot3+1)\cdot4+2=6$; plane $i=0$ covers $0,\ldots,11$ and plane $i=1$ covers $12,\ldots,23$.
- **Check.** The next two cells assert every offset, the table, and every round trip.
- **Where this leads.** Part II writes these maps in plain Python before any Triton kernel appears.

In [ ]:
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
M, N = A.shape

offset_2d = lambda i, j: i * N + j
unflatten_2d = lambda f: (f // N, f % N)

assert A.shape == (3, 4)
assert A.tolist() == [[0.0, 1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0, 7.0],
                      [8.0, 9.0, 10.0, 11.0]]
assert offset_2d(1, 2) == 6 and A[1, 2] == 6
assert (2 * N + torch.arange(N)).tolist() == [8, 9, 10, 11]
assert A[2].tolist() == [8.0, 9.0, 10.0, 11.0]
assert (torch.arange(M) * N + 3).tolist() == [3, 7, 11]
assert A[:, 3].tolist() == [3.0, 7.0, 11.0]
assert A.reshape(-1).tolist() == [float(x) for x in range(12)]
assert unflatten_2d(6) == (1, 2)

expected_2d = [
    (0, 0), (0, 1), (0, 2), (0, 3),
    (1, 0), (1, 1), (1, 2), (1, 3),
    (2, 0), (2, 1), (2, 2), (2, 3),
]
assert [unflatten_2d(f) for f in range(M * N)] == expected_2d
for f in range(M * N):
    r, c = unflatten_2d(f)
    assert offset_2d(r, c) == f, (f, r, c)

D0, D1, D2 = 2, 3, 4
T3 = torch.arange(D0 * D1 * D2, dtype=torch.float32).reshape(D0, D1, D2)
offset_3d = lambda i, j, k: (i * D1 + j) * D2 + k
plane = lambda i: i * D1 * D2 + torch.arange(D1)[:, None] * D2 + torch.arange(D2)[None, :]

assert T3.shape == (2, 3, 4) and T3.stride() == (12, 4, 1)
assert offset_3d(1, 2, 3) == 23 and T3[1, 2, 3] == 23
assert offset_3d(0, 1, 2) == 6 and T3[0, 1, 2] == 6
assert plane(0).tolist() == [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]
assert plane(1).tolist() == [[12, 13, 14, 15], [16, 17, 18, 19], [20, 21, 22, 23]]

In [ ]:
def test_worked_example() -> None:
    '''Assert every offset and round trip stated in the worked example.'''
    A = torch.arange(12).reshape(3, 4)
    M, N = A.shape

    offset_2d = lambda i, j: i * N + j
    unflatten_2d = lambda f: (f // N, f % N)

    assert offset_2d(1, 2) == 6
    assert int(A[1, 2]) == 6

    assert (2 * N + torch.arange(N)).tolist() == [8, 9, 10, 11]
    assert (torch.arange(M) * N + 3).tolist() == [3, 7, 11]
    assert A.reshape(-1).tolist() == list(range(12))

    assert unflatten_2d(6) == (1, 2)
    for f in range(M * N):
        r, c = unflatten_2d(f)
        assert offset_2d(r, c) == f, (f, r, c)

    expected = [
        (0, 0), (0, 1), (0, 2), (0, 3),
        (1, 0), (1, 1), (1, 2), (1, 3),
        (2, 0), (2, 1), (2, 2), (2, 3),
    ]
    assert [unflatten_2d(f) for f in range(12)] == expected

    D0, D1, D2 = 2, 3, 4
    T3 = torch.arange(D0 * D1 * D2).reshape(D0, D1, D2)
    offset_3d = lambda i, j, k: (i * D1 + j) * D2 + k

    assert T3.stride() == (12, 4, 1)
    assert offset_3d(1, 2, 3) == 23
    assert int(T3[1, 2, 3]) == 23
    assert offset_3d(0, 1, 2) == 6

    for i in range(D0):
        for j in range(D1):
            for k in range(D2):
                f = offset_3d(i, j, k)
                ii = f // (D1 * D2)
                jj = (f % (D1 * D2)) // D2
                kk = f % D2
                assert (ii, jj, kk) == (i, j, k), (i, j, k, f)

    print("worked example: 2-D offsets, row/column, and 3-D round trips match")


test_worked_example()

# Part II — Method 01: host-side index arithmetic

- **Connection.** Turns the formulas of Part I into the reference implementation.
- **Rule.** Plain Python integers, one loop per element, no Triton.
- **Flatten and unflatten.** $f=iN+j$ and $i=f//N$, $j=f\bmod N$ are inverses.
- **Row gather.** $Y_{r,:}=X_{I_r,:}$ copies rows in an arbitrary order.
- **Column gather.** $Z_{:,c}=X_{:,J_c}$ copies columns in an arbitrary order.
- **Where.** These are the host versions of the two Triton gather kernels in Part III.
- **Why slow.** Every element is a Python iteration, but the result is unambiguous and is the correctness oracle.

In [ ]:
def flatten_index(i: int, j: int, width: int) -> int:
    '''Row-major flat offset of (i, j) in a matrix with `width` columns.'''
    return i * width + j


def unflatten_index(flat: int, width: int):
    '''Inverse of flatten_index: recover (row, col) from a flat offset.'''
    return flat // width, flat % width


def flatten_index3(i: int, j: int, k: int, d1: int, d2: int) -> int:
    '''Row-major flat offset for shape (D0, d1, d2).'''
    return (i * d1 + j) * d2 + k


def unflatten_index3(flat: int, d1: int, d2: int):
    '''Inverse of flatten_index3: recover (i, j, k).'''
    plane = d1 * d2
    i = flat // plane
    rem = flat % plane
    return i, rem // d2, rem % d2


def naive_row_gather(x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    '''Method 01: Y[r, :] = X[idx[r], :], one element at a time.'''
    out = torch.empty(idx.numel(), x.shape[1], dtype=x.dtype, device=x.device)
    for r in range(idx.numel()):
        src = int(idx[r])
        for c in range(x.shape[1]):
            out[r, c] = x[src, c]
    return out


def naive_col_gather(x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    '''Method 01: Z[:, c] = X[:, idx[c]], one element at a time.'''
    out = torch.empty(x.shape[0], idx.numel(), dtype=x.dtype, device=x.device)
    for c in range(idx.numel()):
        src = int(idx[c])
        for r in range(x.shape[0]):
            out[r, c] = x[r, src]
    return out


A = torch.arange(12).reshape(3, 4)
assert flatten_index(1, 2, 4) == 6
assert unflatten_index(6, 4) == (1, 2)
assert flatten_index3(1, 2, 3, 3, 4) == 23
assert unflatten_index3(23, 3, 4) == (1, 2, 3)

row_idx = torch.tensor([2, 0, 2, 1])
col_idx = torch.tensor([3, 0, 2])
assert naive_row_gather(A, row_idx).tolist() == [
    [8, 9, 10, 11], [0, 1, 2, 3], [8, 9, 10, 11], [4, 5, 6, 7]
]
assert naive_col_gather(A, col_idx).tolist() == [
    [3, 0, 2], [7, 4, 6], [11, 8, 10]
]
print("method 01 reference: flatten/unflatten and both gathers checked")

In [ ]:
def test_host_reference() -> None:
    '''CPU test: flatten/unflatten round trips and the two gather references.'''
    A = torch.arange(12).reshape(3, 4)
    M, N = A.shape

    for i in range(M):
        for j in range(N):
            f = flatten_index(i, j, N)
            assert unflatten_index(f, N) == (i, j), (i, j, f)

    T3 = torch.arange(24).reshape(2, 3, 4)
    for i in range(2):
        for j in range(3):
            for k in range(4):
                f = flatten_index3(i, j, k, 3, 4)
                assert unflatten_index3(f, 3, 4) == (i, j, k), (i, j, k, f)

    row_idx = torch.tensor([2, 0, 2, 1])
    col_idx = torch.tensor([3, 0, 2])
    assert torch.equal(naive_row_gather(A, row_idx), A[row_idx])
    assert torch.equal(naive_col_gather(A, col_idx), A[:, col_idx])

    assert (torch.arange(M) * N + 3).tolist() == [3, 7, 11]
    print("method 01: flatten/unflatten, row gather, and column gather OK")


test_host_reference()

# Part III — Method 02: 2-D offset tiles, tile copy, and gather

- **Connection.** Builds on 6.2, 6.9, and 6.10.
- **Rule.** A 2-D launch uses $p_m=\texttt{tl.program\_id}(0)$ and $p_n=\texttt{tl.program\_id}(1)$.
- **Grid.** $(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$ programs.
- **Tile.** Tile $(p_m,p_n)$ owns rows $p_mB_M+\{0,\ldots,B_M-1\}$ and columns $p_nB_N+\{0,\ldots,B_N-1\}$.
- **Formula.** $O=\text{rows}[:,None]\cdot N+\text{cols}[None,:]\cdot1$.
- **Tile map.** For $M=5$, $N=7$, $B_M=2$, $B_N=4$ the grid is $(3,2)$, so there are six programs.
- **Edge.** The last row and the last column are dead in the edge tiles.
- **Bridge.** The next cell rebuilds the table and asserts every row, column, and mask.

| pid $(p_m,p_n)$ | rows | cols | mask `(2,4)` |
|---|---|---|---|
| `(0,0)` | 0,1 | 0,1,2,3 | `[[1,1,1,1],[1,1,1,1]]` |
| `(0,1)` | 0,1 | 4,5,6,7 | `[[1,1,1,0],[1,1,1,0]]` |
| `(1,0)` | 2,3 | 0,1,2,3 | `[[1,1,1,1],[1,1,1,1]]` |
| `(1,1)` | 2,3 | 4,5,6,7 | `[[1,1,1,0],[1,1,1,0]]` |
| `(2,0)` | 4,5 | 0,1,2,3 | `[[1,1,1,1],[0,0,0,0]]` |
| `(2,1)` | 4,5 | 4,5,6,7 | `[[1,1,1,0],[0,0,0,0]]` |

In [ ]:
def tile_entries(M: int, N: int, BM: int, BN: int):
    gm, gn = triton.cdiv(M, BM), triton.cdiv(N, BN)
    entries = []
    for pm in range(gm):
        rows_t = pm * BM + torch.arange(BM)
        for pn in range(gn):
            cols_t = pn * BN + torch.arange(BN)
            mask = (rows_t[:, None] < M) & (cols_t[None, :] < N)
            entries.append((pm, pn, rows_t.tolist(), cols_t.tolist(),
                            mask.int().tolist()))
    return (gm, gn), entries


grid, entries = tile_entries(5, 7, 2, 4)
assert grid == (3, 2)
assert entries == [
    (0, 0, [0, 1], [0, 1, 2, 3], [[1, 1, 1, 1], [1, 1, 1, 1]]),
    (0, 1, [0, 1], [4, 5, 6, 7], [[1, 1, 1, 0], [1, 1, 1, 0]]),
    (1, 0, [2, 3], [0, 1, 2, 3], [[1, 1, 1, 1], [1, 1, 1, 1]]),
    (1, 1, [2, 3], [4, 5, 6, 7], [[1, 1, 1, 0], [1, 1, 1, 0]]),
    (2, 0, [4, 5], [0, 1, 2, 3], [[1, 1, 1, 1], [0, 0, 0, 0]]),
    (2, 1, [4, 5], [4, 5, 6, 7], [[1, 1, 1, 0], [0, 0, 0, 0]]),
]

## The tile-copy kernel

- **Connection.** Exercises 6.2, 6.9, 6.10, and 6.11 at once.
- **Rule.** It is the identity map written with explicit 2-D offsets.
- **Triton.** The whole address is one `rows[:, None] * N + cols[None, :]` tile.

```python
rows = pid_m * BM + tl.arange(0, BM)
cols = pid_n * BN + tl.arange(0, BN)
mask = (rows[:, None] < M) & (cols[None, :] < N)
offs = rows[:, None] * N + cols[None, :]
x = tl.load(x_ptr + offs, mask=mask, other=0.0)
tl.store(out_ptr + offs, x, mask=mask)
```

- **Check.** The load and the store use the same offsets and the same mask.
- **Failure it exposes.** Any error in the offset formula returns a permuted or shifted tensor.

In [ ]:
@triton.jit
def tile_copy_kernel(x_ptr, out_ptr, M, N,
                     BM: tl.constexpr, BN: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BM + tl.arange(0, BM)
    cols = pid_n * BN + tl.arange(0, BN)

    # The mask is rank 2, exactly like the offset tile.
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    offs = rows[:, None] * N + cols[None, :]

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x, mask=mask)


def tile_copy(x, BM: int = 2, BN: int = 4):
    '''Method 02: copy a 2-D tensor with a 2-D grid of (BM, BN) tiles.'''
    x = torch.as_tensor(x).contiguous()
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BM), triton.cdiv(N, BN))
    tile_copy_kernel[grid](x, out, M, N, BM=BM, BN=BN)
    return out


def test_tile_copy() -> None:
    require_gpu("ch06 tile copy")
    torch.manual_seed(0)

    # 5x7 is not a multiple of 2x4: every edge tile is exercised.
    x = torch.randn(5, 7, device=DEVICE)
    check("tile copy 5x7 BM=2 BN=4", tile_copy(x, BM=2, BN=4), x)

    # A shape that is exactly one tile.
    y = torch.randn(2, 4, device=DEVICE)
    check("tile copy 2x4 single tile", tile_copy(y, BM=2, BN=4), y)

    # A shape that is one row and one column over a tile boundary.
    z = torch.randn(3, 5, device=DEVICE)
    check("tile copy 3x5", tile_copy(z, BM=2, BN=4), z)
    print("tile copy: offsets and 2-D masks agree with PyTorch")


run_if_gpu(test_tile_copy)

## Row gather and column gather

- **Connection.** Reuses the row formula of 6.4 and the column formula of 6.5.
- **Rule.** A gather replaces part of the address with a value loaded from an index array.
- **Row gather.** Program $r$ loads $I_r$ and writes output row $r$ at address $\text{src}\cdot N+j$ for $j=0,\ldots,N-1$.
- **Column gather.** Program $c$ loads $J_c$ and writes output column $c$ at address $i\cdot N+\text{src}$ for $i=0,\ldots,M-1$.
- **Where.** Row gather reads and writes contiguous rows; column gather touches one element per cache line, so it is correct but not the fast path.
- **Output stride.** A column gather has output shape $(M,K)$ and is contiguous, so its row stride is $K$, not $N$.
- **Failure.** Using $N$ for the output stride writes past the end of the buffer.

In [ ]:
@triton.jit
def row_gather_kernel(x_ptr, idx_ptr, out_ptr, n_rows, N,
                      BLOCK_N: tl.constexpr):
    r = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = cols < N

    src = tl.load(idx_ptr + r)
    v = tl.load(x_ptr + src * N + cols, mask=mask, other=0.0)
    tl.store(out_ptr + r * N + cols, v, mask=mask)


def row_gather(x, idx):
    '''Method 02: Y[r, :] = X[idx[r], :]; one program per output row.'''
    x = torch.as_tensor(x).contiguous()
    idx = torch.as_tensor(idx).contiguous()
    out = torch.empty(idx.numel(), x.shape[1], dtype=x.dtype, device=x.device)
    row_gather_kernel[(idx.numel(),)](
        x, idx, out, idx.numel(), x.shape[1],
        BLOCK_N=triton.next_power_of_2(x.shape[1]),
    )
    return out


@triton.jit
def col_gather_kernel(x_ptr, idx_ptr, out_ptr, M, N, K,
                      BLOCK_M: tl.constexpr):
    c = tl.program_id(0)
    rows = tl.arange(0, BLOCK_M)
    mask = rows < M

    src = tl.load(idx_ptr + c)
    v = tl.load(x_ptr + rows * N + src, mask=mask, other=0.0)
    # The output is (M, K), so its row stride is K, not N.
    tl.store(out_ptr + rows * K + c, v, mask=mask)


def col_gather(x, idx):
    '''Method 02: Z[:, c] = X[:, idx[c]]; one program per output column.'''
    x = torch.as_tensor(x).contiguous()
    idx = torch.as_tensor(idx).contiguous()
    K = idx.numel()
    out = torch.empty(x.shape[0], K, dtype=x.dtype, device=x.device)
    col_gather_kernel[(K,)](
        x, idx, out, x.shape[0], x.shape[1], K,
        BLOCK_M=triton.next_power_of_2(x.shape[0]),
    )
    return out


def test_gathers() -> None:
    require_gpu("ch06 row/column gather")
    torch.manual_seed(0)
    x = torch.randn(5, 7, device=DEVICE)

    row_idx = torch.tensor([2, 0, 4, 1], device=DEVICE)
    row_out = row_gather(x, row_idx)
    assert row_out.shape == (4, 7) and row_out.stride() == (7, 1)
    check("row gather", row_out, x[row_idx])

    col_idx = torch.tensor([3, 0, 6], device=DEVICE)
    col_out = col_gather(x, col_idx)
    assert col_out.shape == (5, 3) and col_out.stride() == (3, 1)
    check("column gather", col_out, x[:, col_idx])

    # Repeated indices are legal in a gather; this one duplicates row 2.
    dup = torch.tensor([2, 2, 0], device=DEVICE)
    check("row gather duplicates", row_gather(x, dup), x[dup])
    print("gathers: row-major and strided-column addresses agree with PyTorch")


run_if_gpu(test_gathers)

## Broadcasting offsets: inside the kernel, not on the host

- **Connection.** Builds on 6.9 and the two index vectors of 6.1.
- **Rule.** `rows[:, None]` turns $(B_M,)$ into $(B_M,1)$; `cols[None, :]` turns $(B_N,)$ into $(1,B_N)$.
- **Formula.** Adding them broadcasts the size-1 axis and produces $(B_M,B_N)$.
- **Triton.** `tl.expand_dims(x, axis)` is the same rank change written as a function.
- **Not host broadcasting.** A kernel receives raw pointers and cannot discover an expanded view.

| | Host (PyTorch) | Device (Triton) |
|---|---|---|
| What broadcasts | tensor values, right-aligned | tile values in an expression |
| Shape source | tensor metadata | the tile's own shape |
| Stride source | tensor strides, possibly 0 | passed as scalar arguments |
| Output | a view or a materialised tensor | a register tile |

- **Rule of thumb.** Build the offset tile inside the kernel and pass strides as scalars.
- **Failure.** `rows + cols` with two rank-1 vectors is a vector add, not an outer sum; it is silent when the lengths match and raises when they differ.

In [ ]:
rows = torch.arange(3)[:, None]      # (3, 1)
cols = torch.arange(4)[None, :]      # (1, 4)
assert rows.shape == (3, 1) and rows.stride() == (1, 1)
assert cols.shape == (1, 4) and cols.stride() == (4, 1)
assert torch.broadcast_shapes(rows.shape, cols.shape) == (3, 4)
assert (rows + cols).tolist() == [[0, 1, 2, 3],
                                  [1, 2, 3, 4],
                                  [2, 3, 4, 5]]

# Without the rank change, the same expression is a vector operation.
same = torch.arange(3) + torch.arange(3)
assert same.shape == (3,)
try:
    torch.arange(3) + torch.arange(4)
except RuntimeError:
    pass
else:
    raise AssertionError("1-D rows + cols must raise for lengths 3 and 4")

assert (rows * 4 + cols).tolist() == [[0, 1, 2, 3],
                                      [4, 5, 6, 7],
                                      [8, 9, 10, 11]]

# The same construction inside a kernel, with tl.expand_dims.
# BM and BN must be powers of two; M and N are padded with a rank-2 mask.
@triton.jit
def broadcast_offset_kernel(out_ptr, M, N,
                            BM: tl.constexpr, BN: tl.constexpr):
    r = tl.arange(0, BM)
    c = tl.arange(0, BN)
    off = tl.expand_dims(r, 1) * N + tl.expand_dims(c, 0)
    mask = (r[:, None] < M) & (c[None, :] < N)
    tl.store(out_ptr + off, off, mask=mask)


def broadcast_offsets(M: int, N: int):
    out = torch.empty(M, N, dtype=torch.int32, device=DEVICE)
    broadcast_offset_kernel[(1,)](
        out, M, N,
        BM=triton.next_power_of_2(M), BN=triton.next_power_of_2(N),
    )
    return out


def test_broadcast_offsets() -> None:
    require_gpu("ch06 broadcast offsets")
    ref = torch.arange(12, dtype=torch.int32, device=DEVICE).reshape(3, 4)
    check("tl.expand_dims offset tile", broadcast_offsets(3, 4), ref)
    print("broadcast: [:,None]/[None,:] build a tile; they do not read strides")


run_if_gpu(test_broadcast_offsets)

# Part IV — Method 03: 3-D addresses, strides, and an applied row scale

- **Connection.** Combines Part I's 3-D formula with the host reference of Part II and the kernels of Part III.
- **Where.** A 3-D copy recovers $(i,j,k)$ from a flat program id and multiplies by three strides; a row-wise scale uses 6.4 with a scalar broadcast; a `tl.static_range` kernel shows the arbitrary-rank sum unrolled.
- **Triton.** The destination is contiguous, so its offset is just $f$; the source may be contiguous, permuted, or sliced.

## 3-D copy with stride arguments

- **Rule.** Given a flat lane $f$, recover $i=f//(D_1D_2)$, $j=(f\bmod D_1D_2)//D_2$, and $k=f\bmod D_2$.
- **Formula.** $\text{src}=i\,s_0+j\,s_1+k\,s_2$ and $\text{dst}=f$.
- **Where.** The source strides are passed by the host, so the same kernel serves a permuted input.
- **Failure.** Cast the flat index to `int64` when $MN$ can exceed $2^{31}-1$.

In [ ]:
@triton.jit
def copy3d_kernel(x_ptr, out_ptr, n_elements, D1, D2,
                  s0, s1, s2, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    flat = (pid * BLOCK + tl.arange(0, BLOCK)).to(tl.int64)
    mask = flat < n_elements

    i = flat // (D1 * D2)
    rem = flat % (D1 * D2)
    j = rem // D2
    k = rem % D2

    src = i * s0 + j * s1 + k * s2
    x = tl.load(x_ptr + src, mask=mask, other=0.0)
    tl.store(out_ptr + flat, x, mask=mask)


def copy3d(x, BLOCK: int = 8):
    '''Method 03: copy a rank-3 tensor using its element strides.'''
    x = torch.as_tensor(x)
    D0, D1, D2 = x.shape
    out = torch.empty(D0, D1, D2, dtype=x.dtype, device=x.device)
    copy3d_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), D1, D2, *x.stride(), BLOCK=BLOCK
    )
    return out


def test_copy3d() -> None:
    require_gpu("ch06 3-D copy")
    T = torch.arange(24, device=DEVICE, dtype=torch.float32).reshape(2, 3, 4)
    check("copy3d contiguous", copy3d(T), T)

    P = T.permute(1, 0, 2)
    assert P.shape == (3, 2, 4) and P.stride() == (4, 12, 1)
    check("copy3d permuted", copy3d(P), P.contiguous())
    print("copy3d: flat recovery and stride arguments agree with PyTorch")


run_if_gpu(test_copy3d)

## Applied example: row-wise scale

- **Connection.** Uses row indexing (6.4) and a scalar broadcast (6.9).
- **Rule.** One program owns one row.
- **Formula.** $Y_{ij}=X_{ij}\,s_i$.
- **Triton.** The row offset is 6.4 and the scale is a scalar load that broadcasts across the row tile.

```python
row = tl.program_id(0)
cols = tl.arange(0, BLOCK_N)
mask = cols < N
s = tl.load(scale_ptr + row)
x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)
tl.store(out_ptr + row * N + cols, x * s, mask=mask)
```

- **Example.** `BLOCK_N` is rounded up to a power of two, so $N=7$ uses `BLOCK_N=8` and the mask kills the eighth lane.
- **Failure.** This is the same tail problem as 6.1, one rank up.

In [ ]:
@triton.jit
def row_scale_kernel(x_ptr, scale_ptr, out_ptr, N,
                     BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = cols < N

    s = tl.load(scale_ptr + row)
    x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)
    tl.store(out_ptr + row * N + cols, x * s, mask=mask)


def row_scale(x, scale):
    '''Method 03: Y[i, :] = X[i, :] * scale[i].'''
    x = torch.as_tensor(x).contiguous()
    scale = torch.as_tensor(scale).contiguous()
    out = torch.empty_like(x)
    row_scale_kernel[(x.shape[0],)](
        x, scale, out, x.shape[1],
        BLOCK_N=triton.next_power_of_2(x.shape[1]),
    )
    return out


def test_row_scale() -> None:
    require_gpu("ch06 row scale")
    assert triton.next_power_of_2(7) == 8
    torch.manual_seed(0)
    x = torch.randn(5, 7, device=DEVICE)
    s = torch.randn(5, device=DEVICE)
    check("row scale 5x7", row_scale(x, s), x * s[:, None])

    y = torch.randn(3, 4, device=DEVICE) + 1.0
    t = torch.arange(3, device=DEVICE, dtype=y.dtype) + 1.0
    check("row scale 3x4", row_scale(y, t), y * t[:, None])
    print("row scale: row offset 6.4 and scalar broadcast agree with PyTorch")


run_if_gpu(test_row_scale)

## Arbitrary dimensions with `tl.static_range`

- **Connection.** Generalises 6.12 and the 3-D copy above.
- **Rule.** For rank 3, $o=(bD_1+i)D_2+k=b(D_1D_2)+iD_2+k$.
- **Triton.** `for b in tl.static_range(0, D0)` unrolls at compile time, so $D_0$ must be a `tl.constexpr`.
- **Where.** Each iteration writes one $(B_M,B_N)$ plane; nesting one more `static_range` extends the pattern to rank 4.
- **Why not a runtime `for`.** A runtime loop still works for a copy but prevents constant-folding of the plane offset and the bounds.
- **Failure.** A runtime integer as the `static_range` bound raises; one `BM` for a larger dimension silently truncates unless the mask $i\lt D_1$ is present.

In [ ]:
@triton.jit
def static_range_offset_kernel(out_ptr, D1, D2, D0: tl.constexpr,
                               BM: tl.constexpr, BN: tl.constexpr):
    m = tl.arange(0, BM)
    n = tl.arange(0, BN)
    mask = (m[:, None] < D1) & (n[None, :] < D2)
    for b in tl.static_range(0, D0):
        off = (b * D1 + m[:, None]) * D2 + n[None, :]
        tl.store(out_ptr + off, off, mask=mask)


def static_offsets(D0: int, D1: int, D2: int, BM: int, BN: int):
    out = torch.empty(D0, D1, D2, dtype=torch.int32, device=DEVICE)
    static_range_offset_kernel[(1,)](out, D1, D2, D0=D0, BM=BM, BN=BN)
    return out


def test_static_offsets() -> None:
    require_gpu("ch06 static_range offsets")
    ref = torch.arange(24, dtype=torch.int32, device=DEVICE).reshape(2, 3, 4)
    # D1=3 needs BM=4; the fourth row is masked.
    check("static_range 3-D offsets", static_offsets(2, 3, 4, BM=4, BN=4), ref)
    print("static_range: D0=2 unrolled; BM=4 covers D1=3 with one masked row")


run_if_gpu(test_static_offsets)

# Pitfalls, checks, and summary

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Row and column swapped | $o=jN+i$ transposes the result silently | write $o=iN+j$ and test a non-square shape |
| Missing `pid * B` | every program reads the first tile | $o=\text{pid}\cdot B+i$ |
| Off-by-one at the tail | the last lane reads or writes out of bounds | $o\lt N$, not $o\le N$; use `cdiv` for the grid |
| 2-D mask of rank 1 | `(BM,)` or `(BN,)` broadcasts over one axis and lets the other run out of bounds | `(rows[:,None]<M) & (cols[None,:]<N)` |
| `rows + cols` | a vector add, not an outer sum; silent when the lengths match | expand with `[:,None]` and `[None,:]` first |
| int32 overflow | offsets wrap for tensors larger than $2^{31}-1$ elements | compute the flat index in `int64` before multiplying |
| Flatten with the wrong width | `f // W` recovers the wrong row for every entry | use $N$, the number of columns, not `numel()` |
| Flatten order | column-major uses $f=jM+i$ | state the layout; use `.contiguous()` and C-order |
| Non-contiguous input | hard-coded $N$ reads the wrong element | pass `x.stride()` and use $o=i s_0+j s_1$ |
| 3-D nesting order | $iD_0D_1+\cdots$ reads the wrong plane | $(iD_1+j)D_2+k$ for contiguous C-order |
| Byte stride mixed with element stride | every address is scaled by `element_size()` | pass `x.stride()`, not `x.stride()*x.element_size()` |
| Output stride equals input stride | column gather writes past `(M,K)` when $K\ne N$ | use the output's own stride, $K$ |

## Self-check

1. For a $3\times4$ matrix, compute $o(2,1)$ and recover $(i,j)$ from $f=9$.
2. What are the contiguous strides of $(2,3,4)$? What is $o(1,2,3)$?
3. Why does `rows[:, None]` have shape `(BM,1)`, and what shape is `rows[None, :]`?
4. For $M=5$, $N=7$, $B_M=2$, $B_N=4$, how many programs are launched, and which tiles have dead lanes?
5. A mask has shape `(2,)` for a `(2,4)` tile. What does it guard, and what does it miss?
6. Write the row offset and the column offset for a $5\times7$ matrix. Which one is strided?
7. Why is the output row stride of a column gather $K$, not $N$?
8. When must a flat offset be promoted to `int64`?
9. Why is host broadcasting not the same as `rows[:, None] + cols[None, :]`?
10. What fails if $D_0$ in `tl.static_range(0, D_0)` is a runtime argument?

## Summary

- An address is the dot product of a coordinate with the element strides: $o=\sum_k i_k s_k$.
- 1-D, 2-D, and 3-D offsets are the same formula at ranks 1, 2, and 3.
- Row indexing is contiguous (stride 1); column indexing is strided (stride $N$).
- Flatten is $f=iN+j$; unflatten is $i=f//N$, $j=f\%N$.
- `rows[:, None]` and `cols[None, :]` build an offset tile; the mask must have the same tile shape.
- A tiled launch uses a 2-D grid and `cdiv`; the edge tiles are guarded by a rank-2 mask.
- Pass element strides, not byte strides. Promote large flat offsets to `int64`.
- `tl.static_range` unrolls a compile-time loop and makes the arbitrary-rank formula explicit.

**Where this leads.** Chapter 7 lifts these index maps into elementwise arithmetic and the `tl.load → f → tl.store` skeleton.

**Next:** [Chapter 7 — Elementwise Arithmetic](../../part3_math/ch07_arithmetic/07_complete.ipynb)

# Exercises

1. **Offset table.** For a $4\times5$ matrix, write the full table of $f$, $(i,j)$, and $o(i,j)$. Verify it with a loop and with `torch.arange(20).reshape(4,5)`.
2. **Row versus column.** Write both gathers for a $6\times4$ matrix with the index vector $(3,0,3,1)$. Which kernel reads the larger number of cache lines, and why?
3. **Tile map.** Repeat the tile-map table for $M=7$, $N=5$, $B_M=4$, $B_N=2$. Mark every dead lane.
4. **Mask rank bug.** Change the 2-D mask to `(rows < M)` only. Run a $5\times7$ copy and describe the failure. Which access fails first?
5. **Strided copy.** Pass the transpose of a $4\times6$ tensor to `copy3d` (add a leading batch axis) and check the result against `.contiguous()`.
6. **Column-major.** Change the flattening convention to $f=jM+i$. Write the new unflatten formula and update the worked example.
7. **int64.** Making a tensor with more than $2^{31}$ elements is impractical; instead, force `flat` to stay `int32` in `copy3d_kernel` and assert the widest offset for a large shape. Explain where it would wrap.
8. **Arbitrary rank.** Extend `static_range_offset_kernel` to rank 4 by adding a second `tl.static_range`. Test it against `torch.arange(2*3*4*5).reshape(2,3,4,5)`.
9. **Row-wise softmax sketch.** Using row indexing only, write the address expression for a row-wise max followed by a row-wise sum. Do not implement the full softmax; just write the two offset vectors.
10. **Traffic.** For a $1024\times1024$ `float32` copy, compare a row-wise kernel with a 2-D tiled kernel in terms of loads, stores, and bytes moved. Assume both are correct.

- Hints are in [`../../docs/SOLUTIONS.md`](../../docs/SOLUTIONS.md).

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_host_reference()

print("\nRunning GPU/interpreter tests (skipped without CUDA/ROCm) ...")
for fn in (
    test_tile_copy,
    test_gathers,
    test_broadcast_offsets,
    test_copy3d,
    test_row_scale,
    test_static_offsets,
):
    run_if_gpu(fn)